# Ventilation of two zones: non-linear models

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/cghiaus/dm4bem_book/HEAD?labpath=%2Ftutorials%2Ff06_flow_2zone_non_lin.ipynb)

This sections deals with a multizone model characterized by non-linear constitutuive equations. The problem is solved by using the functions `dm4bem.flow_network_fixed_point()` and `dm4bem.flow_network_least_squares()`. The reference level for the indoor air pressures and the initial guess of the direction of mass flow rates does ot change the final results.

## Problem

Let's consider the ventilation of two zones (Figure 1) in the case of non-linear constitutive law:

$$ \dot V_{i,j} = K_{i,j} \, \Delta P^n_{i,j} = K_{i,j} \, (P_i - P_j)^n$$

$$ \dot m_{i,j} = \rho_{up, i} \, \dot V_{i,j}$$

![vent_2z](../figures/f05_vent_2z_principe.svg)
> Figure 1. Ventilation of two zones.

The pressure sources are:
- stack pressure: $P_s = P_0 - \rho \, g \, z$;
- wind pressure: $P_w = C_p \frac{1}{2} \rho_o \, v^2$

The flow source is mechanical controlled ventilation.

The problem is solved by __fixed point method__ and by __non linear least squares method__.

The flow directions and the reference height for the indoor pressures are changed from one example to another. Note the changes in the incidence matrix $A$ and in the pressure source vector $b$. The values in pressure sources vector $b$ follow the [source sign convention](https://en.wikipedia.org/wiki/Passive_sign_convention): the source is positive if the flow is from $-$ to $+$ and negative if the flow is reversed.

## Data

In [1]:
import numpy as np

import dm4bem

# =================================================
# Data
# =================================================

g = 10.0                         # m/s², gravity
rho_e, rho_i = 1.3, 1.2          # kg/m³, density: outdoor, indoor

z0, z1, z2 = 1, 1.5, 2.5         # m, orrifice height
K0, K1, K2 = 50, 20, 50          # m³/(h·Paⁿ), flow coefficients
n0, n1, n2 = 0.5, 0.65, 0.8      # pressure exponent
Cpn, Cps = 0.9, -0.1             # wind pressure coefficients on North and South facades
V_dot_VMC = -200                 # m³/h, mechanical ventilation extraction

v = 3                            # m/s, wind velocity

# =================================================
# Calculations
# =================================================

# Static pressures, Pa
Ps0 = -rho_e * g * z0
Psi0 = -rho_i * g * (z1 - z0)
Psi1 = -rho_i * g * (z2 - z1)
Ps2 = -rho_e * g * z2

# Dynamic pressures, Pa
Pd0 = Cpn * 0.5 * rho_e * v**2
Pd2 = Cps * 0.5 * rho_e * v**2

m_dot_MV = rho_i * V_dot_VMC / 3600

# Mass-flow sources, kg/s
f = np.array([0, m_dot_MV])

# Nodes densities
rho_out = rho_e
rho = np.array([rho_i, rho_i])

# Flow coefficients in m³/(s·Paⁿ)
K = np.array([K0, K1, K2]) / 3600

# Pressure exponents
n = np.array([n0, n1, n2])

# Initial pressure guess, Pa
p0 = np.array([-10.0, -22.0])

def print_results():
    print(f"Pressures: {p.round(2)} Pa")
    print(f"Mass flows: {(m_dot * 3600).round(2)} kg/h")
    print(f"Volume flows: {(V_dot * 3600).round(2)} m³/h")
    
    # Verify mass conservation
    print(f"Error in mass balance: {A.T @ m_dot + f}kg/s")

## Solutions for different reference pressures and flow orientation

### Pressures at z0 and z1, m2 normal

![ex_a](../figures/f06_flow_net_ex_a.svg)
> Figure 2. Ventilation of two zones: $p_{i0}$ at $z_0$, $p_{i1}$ at $z_1$

In [2]:
# Network topology
A = np.array([
    [ 1, 0],
    [-1, 1],
    [ 0, 1]])

# Pressure sources (case a)
b = np.array([Ps0 + Pd0,            # branch 0
              Psi0,                 # branch 1
              Ps2 + Pd2 - Psi1])    # branch 2 (Psi1 < 0)

In [3]:
p, m_dot, V_dot = dm4bem.flow_network_fixed_point(
    A, b, f, K, n, rho, rho_out, p0)

print_results()

Pressures: [ -9.79 -23.85] Pa
Mass flows: [ 93.2  93.2 146.8] kg/h
Volume flows: [ 71.69  77.67 112.92] m³/h
Error in mass balance: [ 5.34775026e-10 -5.60574642e-11]kg/s


In [4]:
p, m_dot, V_dot = dm4bem.flow_network_least_squares(
    A, b, f, K, n, rho, rho_out, p0)

print_results()

Pressures: [ -9.79 -23.85] Pa
Mass flows: [ 93.2  93.2 146.8] kg/h
Volume flows: [ 71.69  77.67 112.92] m³/h
Error in mass balance: [6.93889390e-18 1.38777878e-17]kg/s


### Pressures at z0 and z1, m2 reversed

![ex_a](../figures/f06_flow_net_ex_b.svg)
> Figure 3. Ventilation of two zones: $p_{i0}$ at $z_0$, $p_{i1}$ at $z_1$

In [5]:
# Network topology
A = np.array([
    [ 1, 0],
    [ 1, -1],
    [ 0, -1]])

# Pressure sources (case a)
b = np.array([Ps0 + Pd0,                # branch 0
              -Psi0,                    # branch 1 (Psi0 < 0)
              -(Ps2 + Pd2 - Psi1)])     # branch 2 (Ps2 < 0, Pd2 < 0, Psi1 > 0)

In [6]:
p, m_dot, V_dot = dm4bem.flow_network_fixed_point(
    A, b, f, K, n, rho, rho_out, p0)

print_results()

Pressures: [ -9.79 -23.85] Pa
Mass flows: [  93.2  -93.2 -146.8] kg/h
Volume flows: [  71.69  -77.67 -112.92] m³/h
Error in mass balance: [ 5.34775026e-10 -5.60574642e-11]kg/s


In [7]:
p, m_dot, V_dot = dm4bem.flow_network_least_squares(
    A, b, f, K, n, rho, rho_out, p0)

print_results()

Pressures: [ -9.79 -23.85] Pa
Mass flows: [  93.2  -93.2 -146.8] kg/h
Volume flows: [  71.69  -77.67 -112.92] m³/h
Error in mass balance: [6.93889390e-18 1.38777878e-17]kg/s


### Pressures at z1, m2 normal

![ex_a](../figures/f06_flow_net_ex_c.svg)
> Figure 4. Ventilation of two zones: both $p_{i0}$ and $p_{i1}$ at $z_1$

In [8]:
# Network topology
A = np.array([
    [ 1, 0],
    [-1, 1],
    [ 0, 1]])

# Pressure sources (case a)
b = np.array([Ps0 + Pd0 + Psi0,         # branch 0
              0,                        # branch 1
              Ps2 + Pd2 - Psi1])        # branch 2 (Ps1 < 0)

In [9]:
p, m_dot, V_dot = dm4bem.flow_network_fixed_point(
    A, b, f, K, n, rho, rho_out, p0)

print_results()

Pressures: [-15.79 -23.85] Pa
Mass flows: [ 93.2  93.2 146.8] kg/h
Volume flows: [ 71.69  77.67 112.92] m³/h
Error in mass balance: [-4.28809002e-10  4.46201409e-11]kg/s


### Pressures at z1 and z2, m2 reversed

![ex_a](../figures/f06_flow_net_ex_d.svg)
> Figure 5. Ventilation of two zones: $p_{i0}$ at $z_1$, $p_{i1}$ at $z_2$

In [10]:
# Network topology
A = np.array([
    [ 1, 0],
    [-1, 1],
    [ 0, 1]])

# Pressure sources (case a)
b = np.array([Ps0 + Pd0 + Psi0,         # branch 0
              Psi1,                     # branch 1
              Ps2 + Pd2])               # branch 2

In [11]:
p, m_dot, V_dot = dm4bem.flow_network_least_squares(
    A, b, f, K, n, rho, rho_out, p0)

print_results()

Pressures: [-15.79 -35.85] Pa
Mass flows: [ 93.2  93.2 146.8] kg/h
Volume flows: [ 71.69  77.67 112.92] m³/h
Error in mass balance: [-3.46944695e-18 -4.16333634e-17]kg/s
